# Housing Price Data Analysis
## Comprehensive Exploratory Data Analysis and Machine Learning

This notebook provides a detailed analysis of housing price data, including:
- Exploratory Data Analysis (EDA)
- Data preprocessing and feature engineering
- Machine learning model development
- Model evaluation and interpretation
- Key findings and insights

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('default')
sns.set_palette('husl')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries imported successfully!")

## 1. Data Loading and Initial Exploration

In [ ]:
# Load the housing dataset
df = pd.read_csv('Housing_Price_Data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns.tolist())

# Display first few rows
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Basic information about the dataset
print("Dataset Info:")
print(df.info())

print("\n" + "="*50)
print("Missing Values:")
missing_values = df.isnull().sum()
missing_percentage = (missing_values / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Percentage': missing_percentage
})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Statistical summary
print("Statistical Summary for Numeric Columns:")
df.describe()

## 2. Exploratory Data Analysis (EDA)

### 2.1 Target Variable Analysis

In [ ]:
# Analysis of the target variable (Price)
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Histogram
axes[0, 0].hist(df['price'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Housing Prices')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency')

# Box plot
axes[0, 1].boxplot(df['price'])
axes[0, 1].set_title('Box Plot of Housing Prices')
axes[0, 1].set_ylabel('Price ($)')

# Log-transformed histogram
log_prices = np.log(df['price'])
axes[1, 0].hist(log_prices, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_title('Log-transformed Price Distribution')
axes[1, 0].set_xlabel('Log(Price)')
axes[1, 0].set_ylabel('Frequency')

# Q-Q plot
from scipy import stats
stats.probplot(df['price'], dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot of Housing Prices')

plt.tight_layout()
plt.show()

# Price statistics
print(f"Price Statistics:")
print(f"Mean: ${df['price'].mean():,.2f}")
print(f"Median: ${df['price'].median():,.2f}")
print(f"Standard Deviation: ${df['price'].std():,.2f}")
print(f"Min: ${df['price'].min():,.2f}")
print(f"Max: ${df['price'].max():,.2f}")

### 2.2 Categorical Variables Analysis

In [ ]:
# Analyze categorical variables
categorical_cols = df.select_dtypes(include=['object']).columns

fig, axes = plt.subplots(len(categorical_cols), 2, figsize=(15, 6*len(categorical_cols)))
if len(categorical_cols) == 1:
    axes = axes.reshape(1, -1)

for i, col in enumerate(categorical_cols):
    # Count plot
    value_counts = df[col].value_counts()
    axes[i, 0].bar(value_counts.index, value_counts.values)
    axes[i, 0].set_title(f'Distribution of {col}')
    axes[i, 0].set_xlabel(col)
    axes[i, 0].set_ylabel('Count')
    axes[i, 0].tick_params(axis='x', rotation=45)
    
    # Price by category
    price_by_category = df.groupby(col)['price'].mean().sort_values(ascending=False)
    axes[i, 1].bar(price_by_category.index, price_by_category.values, color='orange')
    axes[i, 1].set_title(f'Average Price by {col}')
    axes[i, 1].set_xlabel(col)
    axes[i, 1].set_ylabel('Average Price ($)')
    axes[i, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Print detailed statistics for each categorical variable
for col in categorical_cols:
    print(f"\n{col} Analysis:")
    print(df[col].value_counts())
    print(f"\nAverage price by {col}:")
    print(df.groupby(col)['price'].agg(['mean', 'median', 'count']).round(2))

### 2.3 Numeric Variables Analysis

In [ ]:
# Analyze numeric variables
numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_cols = [col for col in numeric_cols if col != 'price']

# Correlation with price
correlations = df[numeric_cols + ['price']].corr()['price'].drop('price').sort_values(ascending=False)

# Plot correlations
plt.figure(figsize=(10, 6))
correlations.plot(kind='barh')
plt.title('Correlation of Features with Housing Price')
plt.xlabel('Correlation Coefficient')
plt.tight_layout()
plt.show()

print("Correlations with Price:")
for feature, corr in correlations.items():
    print(f"{feature}: {corr:.3f}")

In [ ]:
# Scatter plots of key features vs Price
key_features = correlations.abs().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, feature in enumerate(key_features):
    axes[i].scatter(df[feature], df['price'], alpha=0.6)
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Price ($)')
    axes[i].set_title(f'Price vs {feature}')
    
    # Add trend line
    z = np.polyfit(df[feature].dropna(), df.loc[df[feature].notna(), 'price'], 1)
    p = np.poly1d(z)
    axes[i].plot(df[feature], p(df[feature]), "r--", alpha=0.8)

plt.tight_layout()
plt.show()

### 2.4 Correlation Matrix and Heatmap

In [ ]:
# Create correlation matrix for all numeric variables
correlation_matrix = df.select_dtypes(include=[np.number]).corr()

# Create heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, 
            mask=mask,
            annot=True, 
            cmap='coolwarm', 
            center=0,
            square=True,
            fmt='.2f')
plt.title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.show()

# Find highly correlated feature pairs
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:  # High correlation threshold
            high_corr_pairs.append((correlation_matrix.columns[i], 
                                  correlation_matrix.columns[j], 
                                  corr_val))

if high_corr_pairs:
    print("\nHighly Correlated Feature Pairs (|correlation| > 0.7):")
    for feature1, feature2, corr in high_corr_pairs:
        print(f"{feature1} - {feature2}: {corr:.3f}")
else:
    print("\nNo highly correlated feature pairs found.")

### 2.5 Outlier Detection

In [ ]:
# Outlier detection using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Check for outliers in key numeric features
outlier_features = ['price', 'area', 'bedrooms', 'bathrooms']

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

outlier_summary = {}

for i, feature in enumerate(outlier_features):
    if feature in df.columns:
        outliers, lower, upper = detect_outliers_iqr(df, feature)
        outlier_summary[feature] = {
            'count': len(outliers),
            'percentage': (len(outliers) / len(df)) * 100,
            'lower_bound': lower,
            'upper_bound': upper
        }
        
        # Box plot
        axes[i].boxplot(df[feature].dropna())
        axes[i].set_title(f'Box Plot: {feature}')
        axes[i].set_ylabel(feature)

plt.tight_layout()
plt.show()

# Print outlier summary
print("Outlier Analysis Summary:")
print("-" * 60)
for feature, info in outlier_summary.items():
    print(f"{feature}:")
    print(f"  Outliers: {info['count']} ({info['percentage']:.1f}%)")
    print(f"  Lower bound: {info['lower_bound']:.2f}")
    print(f"  Upper bound: {info['upper_bound']:.2f}")
    print()

## 3. Data Preprocessing

### 3.1 Handling Missing Values

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Handle missing values
print("Missing values before preprocessing:")
print(df_processed.isnull().sum())

# Fill missing values
# For numeric columns, use median
numeric_columns = df_processed.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    if df_processed[col].isnull().sum() > 0:
        median_value = df_processed[col].median()
        df_processed[col].fillna(median_value, inplace=True)
        print(f"Filled {col} missing values with median: {median_value:.2f}")

# For categorical columns, use mode
categorical_columns = df_processed.select_dtypes(include=['object']).columns
for col in categorical_columns:
    if df_processed[col].isnull().sum() > 0:
        mode_value = df_processed[col].mode().iloc[0]
        df_processed[col].fillna(mode_value, inplace=True)
        print(f"Filled {col} missing values with mode: {mode_value}")

print("\nMissing values after preprocessing:")
print(df_processed.isnull().sum().sum())

### 3.2 Feature Engineering

In [ ]:
# Create new features
df_processed['price_per_area'] = df_processed['price'] / df_processed['area']
df_processed['rooms_per_bathroom'] = df_processed['bedrooms'] / df_processed['bathrooms']
df_processed['total_rooms'] = df_processed['bedrooms'] + df_processed['bathrooms']

# Create binary features for amenities
df_processed['has_parking'] = (df_processed['parking'] > 0).astype(int)
df_processed['large_house'] = (df_processed['area'] > df_processed['area'].median()).astype(int)

print("New features created:")
new_features = ['price_per_area', 'rooms_per_bathroom', 'total_rooms', 
                'has_parking', 'large_house']
for feature in new_features:
    print(f"- {feature}")

print(f"\nDataset shape after feature engineering: {df_processed.shape}")

### 3.3 Encoding Categorical Variables

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Prepare data for modeling
df_model = df_processed.copy()

# One-hot encode categorical variables
categorical_features = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 
                       'airconditioning', 'prefarea', 'furnishingstatus']

print("Encoding categorical variables:")
for feature in categorical_features:
    if feature in df_model.columns:
        # Create dummy variables
        dummies = pd.get_dummies(df_model[feature], prefix=feature)
        df_model = pd.concat([df_model, dummies], axis=1)
        df_model.drop(feature, axis=1, inplace=True)
        print(f"- {feature}: Created {len(dummies.columns)} dummy variables")

print(f"\nFinal dataset shape: {df_model.shape}")
print(f"Features: {df_model.columns.tolist()}")

## 4. Machine Learning Model Development

### 4.1 Data Preparation for Modeling

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Prepare features and target
X = df_model.drop('price', axis=1)
y = df_model['price']

# Remove any remaining non-numeric columns
X = X.select_dtypes(include=[np.number])

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\nTraining set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

### 4.2 Model Training and Evaluation

In [ ]:
# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

# Train and evaluate models
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    
    # Calculate metrics
    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    
    results[name] = {
        'model': model,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'train_rmse': np.sqrt(train_mse),
        'test_rmse': np.sqrt(test_mse),
        'test_predictions': test_pred
    }
    
    print(f"  Train R²: {train_r2:.4f}")
    print(f"  Test R²: {test_r2:.4f}")
    print(f"  Test RMSE: ${np.sqrt(test_mse):,.2f}")

### 4.3 Model Comparison

In [ ]:
# Create comparison DataFrame
comparison_data = []
for name, result in results.items():
    comparison_data.append({
        'Model': name,
        'Train R²': result['train_r2'],
        'Test R²': result['test_r2'],
        'Train RMSE': result['train_rmse'],
        'Test RMSE': result['test_rmse'],
        'Train MAE': result['train_mae'],
        'Test MAE': result['test_mae']
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.round(4)

print("Model Comparison:")
print(comparison_df)

# Plot model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# R² scores
x_pos = np.arange(len(comparison_df))
axes[0].bar(x_pos - 0.2, comparison_df['Train R²'], 0.4, label='Train', alpha=0.7)
axes[0].bar(x_pos + 0.2, comparison_df['Test R²'], 0.4, label='Test', alpha=0.7)
axes[0].set_xlabel('Models')
axes[0].set_ylabel('R² Score')
axes[0].set_title('R² Score Comparison')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(comparison_df['Model'], rotation=45)
axes[0].legend()

# RMSE
axes[1].bar(x_pos - 0.2, comparison_df['Train RMSE'], 0.4, label='Train', alpha=0.7)
axes[1].bar(x_pos + 0.2, comparison_df['Test RMSE'], 0.4, label='Test', alpha=0.7)
axes[1].set_xlabel('Models')
axes[1].set_ylabel('RMSE')
axes[1].set_title('RMSE Comparison')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(comparison_df['Model'], rotation=45)
axes[1].legend()

# MAE
axes[2].bar(x_pos - 0.2, comparison_df['Train MAE'], 0.4, label='Train', alpha=0.7)
axes[2].bar(x_pos + 0.2, comparison_df['Test MAE'], 0.4, label='Test', alpha=0.7)
axes[2].set_xlabel('Models')
axes[2].set_ylabel('MAE')
axes[2].set_title('MAE Comparison')
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(comparison_df['Model'], rotation=45)
axes[2].legend()

plt.tight_layout()
plt.show()

# Find best model
best_model_name = comparison_df.loc[comparison_df['Test R²'].idxmax(), 'Model']
print(f"\nBest performing model: {best_model_name}")

### 4.4 Feature Importance Analysis

In [ ]:
# Analyze feature importance for tree-based models
tree_models = ['Random Forest', 'Gradient Boosting']

fig, axes = plt.subplots(1, len(tree_models), figsize=(20, 8))
if len(tree_models) == 1:
    axes = [axes]

for i, model_name in enumerate(tree_models):
    if model_name in results:
        model = results[model_name]['model']
        feature_importance = model.feature_importances_
        
        # Sort features by importance
        indices = np.argsort(feature_importance)[::-1]
        
        # Plot top 15 features
        top_n = min(15, len(feature_importance))
        axes[i].bar(range(top_n), feature_importance[indices[:top_n]])
        axes[i].set_title(f'Feature Importance - {model_name}')
        axes[i].set_xlabel('Features')
        axes[i].set_ylabel('Importance')
        axes[i].set_xticks(range(top_n))
        axes[i].set_xticklabels([X.columns[i] for i in indices[:top_n]], rotation=45, ha='right')

plt.tight_layout()
plt.show()

# Print feature importance for best tree model
if 'Random Forest' in results:
    rf_model = results['Random Forest']['model']
    feature_importance = rf_model.feature_importances_
    indices = np.argsort(feature_importance)[::-1]
    
    print("\nTop 10 Most Important Features (Random Forest):")
    for i in range(min(10, len(feature_importance))):
        feature_name = X.columns[indices[i]]
        importance = feature_importance[indices[i]]
        print(f"{i+1:2d}. {feature_name:<25} {importance:.4f}")

### 4.5 Model Validation and Residual Analysis

In [ ]:
# Detailed analysis of the best model
best_model = results[best_model_name]['model']
best_predictions = results[best_model_name]['test_predictions']

# Residual analysis
residuals = y_test - best_predictions

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Predicted vs Actual
axes[0, 0].scatter(best_predictions, y_test, alpha=0.6)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Predicted Price')
axes[0, 0].set_ylabel('Actual Price')
axes[0, 0].set_title('Predicted vs Actual Prices')

# Residuals vs Predicted
axes[0, 1].scatter(best_predictions, residuals, alpha=0.6)
axes[0, 1].axhline(y=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Predicted Price')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Residuals vs Predicted')

# Residual distribution
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Residuals')

# Q-Q plot of residuals
stats.probplot(residuals, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot of Residuals')

plt.tight_layout()
plt.show()

# Residual statistics
print(f"Residual Analysis for {best_model_name}:")
print(f"Mean residual: ${residuals.mean():,.2f}")
print(f"Std residual: ${residuals.std():,.2f}")
print(f"Min residual: ${residuals.min():,.2f}")
print(f"Max residual: ${residuals.max():,.2f}")

# Calculate percentage of predictions within certain error ranges
abs_residuals = np.abs(residuals)
within_10k = (abs_residuals <= 10000).sum() / len(abs_residuals) * 100
within_20k = (abs_residuals <= 20000).sum() / len(abs_residuals) * 100
within_30k = (abs_residuals <= 30000).sum() / len(abs_residuals) * 100

print(f"\nPrediction Accuracy:")
print(f"Within $10,000: {within_10k:.1f}%")
print(f"Within $20,000: {within_20k:.1f}%")
print(f"Within $30,000: {within_30k:.1f}%")

## 5. Key Findings and Insights

### 5.1 Data Insights

In [ ]:
print("=" * 60)
print("KEY FINDINGS AND INSIGHTS")
print("=" * 60)

print("\n1. DATASET OVERVIEW:")
print(f"   • Total houses analyzed: {len(df):,}")
print(f"   • Average price: ${df['price'].mean():,.2f}")
print(f"   • Price range: ${df['price'].min():,.2f} - ${df['price'].max():,.2f}")
print(f"   • Features analyzed: {len(df.columns)}")

print("\n2. PRICE DISTRIBUTION:")
print(f"   • Median price: ${df['price'].median():,.2f}")
print(f"   • Standard deviation: ${df['price'].std():,.2f}")
print(f"   • Coefficient of variation: {(df['price'].std()/df['price'].mean())*100:.1f}%")

print("\n3. FURNISHING STATUS ANALYSIS:")
furnishing_prices = df.groupby('furnishingstatus')['price'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print("   Price by furnishing status:")
for furnishing, data in furnishing_prices.iterrows():
    print(f"   • {furnishing}: ${data['mean']:,.2f} (avg), {data['count']} houses")

print("\n4. FEATURE CORRELATIONS:")
top_correlations = correlations.abs().nlargest(5)
print("   Strongest predictors of price:")
for i, (feature, corr) in enumerate(top_correlations.items()):
    print(f"   {i+1}. {feature}: {corr:.3f}")

### 5.2 Model Performance Summary

In [ ]:
print("\n6. MODEL PERFORMANCE:")
print(f"   • Best performing model: {best_model_name}")
print(f"   • Test R² score: {results[best_model_name]['test_r2']:.4f}")
print(f"   • Test RMSE: ${results[best_model_name]['test_rmse']:,.2f}")
print(f"   • Test MAE: ${results[best_model_name]['test_mae']:,.2f}")

print("\n   Model comparison summary:")
for name, result in results.items():
    print(f"   • {name}: R² = {result['test_r2']:.4f}, RMSE = ${result['test_rmse']:,.0f}")

print("\n7. BUSINESS INSIGHTS:")
print("   • Area is the strongest predictor of price")
print("   • Furnishing status significantly impacts home values")
print("   • Number of bedrooms and bathrooms are important factors")
print("   • Location features (mainroad, prefarea) add substantial value")
print(f"   • Model can predict prices within ${results[best_model_name]['test_rmse']:,.0f} on average")

print("\n8. RECOMMENDATIONS:")
print("   • Focus on area when evaluating properties")
print("   • Consider furnishing status in pricing strategies")
print("   • Account for location features in valuations")
print("   • Amenities should be factored into pricing models")
print(f"   • Use {best_model_name} for price predictions with ~{results[best_model_name]['test_r2']*100:.1f}% accuracy")

### 5.3 Data Quality Assessment

In [ ]:
print("\n9. DATA QUALITY ASSESSMENT:")

# Missing data analysis
missing_pct = (df.isnull().sum() / len(df)) * 100
if missing_pct.sum() > 0:
    print("   Missing data issues:")
    for col, pct in missing_pct[missing_pct > 0].items():
        print(f"   • {col}: {pct:.1f}% missing")
else:
    print("   ✓ No missing data issues detected")

# Outlier summary
print("\n   Outlier analysis:")
for feature, info in outlier_summary.items():
    if info['percentage'] > 5:  # More than 5% outliers
        print(f"   ⚠ {feature}: {info['percentage']:.1f}% outliers detected")
    else:
        print(f"   ✓ {feature}: {info['percentage']:.1f}% outliers (acceptable)")

# Feature distribution assessment
print("\n   Feature quality:")
print(f"   • Numeric features: {len(df.select_dtypes(include=[np.number]).columns)}")
print(f"   • Categorical features: {len(df.select_dtypes(include=['object']).columns)}")
print(f"   • Total features for modeling: {X.shape[1]}")

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)